# **Chapter 7 — Viscoelasticity and Time-Dependent Mechanics**


<a href="https://colab.research.google.com/github/ronniewillaert/SPM-Textbook-Python/blob/main/notebooks/part-03-nanomechanics/ch07_viscoelasticity/AFM_Ch07_Viscoelasticity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Interactive Companion Notebook

This notebook accompanies **Chapter 7** of the textbook *Atomic Force Microscopy for Bioengineering — From Fundamentals to Quantitative Nanomechanics and Force Spectroscopy*.

The notebook contains **8 interactive exercises**, each with problem statements, runnable code, plots, and interactive sliders, that mirror the Python Exercises listed in Section 7.10.4 of the chapter.

**After completing this notebook you will be able to:**
* Simulate creep and stress-relaxation responses of viscoelastic materials with the Kelvin–Voigt and Maxwell models.
* Compute and interpret the characteristic relaxation time $\tau = \eta / E$.
* Generate oscillatory nanorheology data and extract the storage modulus $E'$, loss modulus $E''$, and loss tangent $\tan\varphi$.
* Build force–deformation Lissajous trajectories and quantify dissipated energy from hysteresis-loop area.
* Distinguish **viscoelastic** from **poroelastic** relaxation using indentation rate and probe size.
* Reproduce typical AFM viscoelastic force curves with rate-dependent stiffness and hysteresis, and fit them with simple models.
* Compute frequency-dependent mechanical spectra for cytoplasm-, cortical-actin-, and hydrogel-like materials.
* Decompose living-cell dynamics into passive viscoelastic and active nonequilibrium (ATP-driven) fluctuations.
* Connect AFM time-domain and frequency-domain observables to the mechanobiology of cells, tissues, and biofilms.

**Learning Outcomes**
* understand why biological materials require **time-dependent** mechanical models rather than purely elastic ones
* derive and interpret the constitutive equations of the Kelvin–Voigt and Maxwell spring–dashpot models
* connect creep, stress-relaxation, and hysteresis to specific AFM experimental protocols
* read and reason about complex moduli $E^{*} = E' + iE''$ and phase angles $\varphi$
* identify the molecular origins of viscoelasticity (polymer rearrangement, fluid flow, bond turnover, active stresses)
* recognize when poroelastic fluid flow — not solid viscosity — controls the relaxation
* relate frequency-dependent spectra to cytoplasmic, cytoskeletal, and tissue mechanics
* appreciate the **finite experimental window** of AFM and how it limits accessible timescales
* link AFM viscoelastic observables to disease-relevant changes in cells and tissues.

---

**How to use this notebook:**
1. Run the first two cells (imports + widgets) before anything else.
2. Each exercise has a *theory cell* (markdown) followed by an *interactive code cell*.
3. Use the sliders to explore — the goal is to build physical intuition, not just run code.
4. At the end of each exercise, there is a short **Reflection** prompt — try to answer it before moving on.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy import stats

# Plotting defaults
plt.rcParams.update({
    'figure.dpi': 100,
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 9,
    'lines.linewidth': 2,
})

# Physical-unit shortcuts
NM   = 1e-9    # nanometre  (m)
UM   = 1e-6    # micrometre (m)
KPA  = 1e3     # kilopascal (Pa)
NN   = 1e-9    # nanonewton (N)
PI   = np.pi


In [ ]:
try:
    from ipywidgets import interact, FloatSlider, IntSlider, FloatLogSlider, Dropdown, Checkbox
except ImportError:
    !pip install ipywidgets -q
    from ipywidgets import interact, FloatSlider, IntSlider, FloatLogSlider, Dropdown, Checkbox


---
## 1. Kelvin–Voigt Creep Simulator — Time-Dependent Strain Under Constant Stress

The **Kelvin–Voigt model** (Section 7.2.2) is a spring (modulus $E$) and a dashpot (viscosity $\eta$) in **parallel**. Its constitutive equation is

$$\sigma(t) = E\,\varepsilon(t) + \eta\,\frac{d\varepsilon}{dt}.$$

Under a sudden, constant stress $\sigma_{0}$ applied at $t = 0$ (an AFM **creep** / force-clamp experiment), the strain rises exponentially toward the equilibrium value $\sigma_{0}/E$:

$$\varepsilon(t) = \frac{\sigma_{0}}{E}\,\Big[\,1 - e^{-t/\tau}\,\Big], \qquad \tau = \frac{\eta}{E}.$$

The **retardation time** $\tau$ is the characteristic timescale over which the material "catches up" with the applied load.

### Exercise

The simulation below applies a step stress $\sigma_{0}$ at $t = 0$ and shows the time evolution of strain. The middle panel sweeps $\eta$ to highlight how the curve shape, but not the final value, depends on viscosity. The right panel marks $\tau$ on a semi-log time axis.

### Tasks
1. Use the default values ($E = 5$ kPa, $\eta = 20$ kPa·s, $\sigma_{0} = 1$ kPa) — this is **Numerical Problem 7.10.2 #1** in the chapter. Verify that $\tau = 4$ s and that $\varepsilon(1\,\text{s}) \approx 0.044$ (≈ 22 % of the equilibrium strain).
2. Hold $E$ fixed and increase $\eta$. Does the equilibrium strain change? Does $\tau$ change?
3. Hold $\eta$ fixed and increase $E$. Now which changes — the equilibrium, the timescale, or both?

**Reflection:** *Why does Kelvin–Voigt always reach an equilibrium strain, while a Maxwell material would creep forever? Which is a better picture of a soft hydrogel — and why?*


In [ ]:
def interactive_kv_creep(E_kPa=5.0, eta_kPa_s=20.0, sigma0_kPa=1.0,
                        t_max_s=30.0, n_points=400):
    """Kelvin-Voigt creep response under a step stress sigma0 applied at t=0."""
    E      = E_kPa * KPA
    eta    = eta_kPa_s * KPA           # Pa·s
    sigma0 = sigma0_kPa * KPA
    tau    = eta / E                    # retardation time (s)

    t      = np.linspace(0, t_max_s, n_points)
    eps    = (sigma0 / E) * (1 - np.exp(-t / tau))
    eps_eq = sigma0 / E

    # Sweep of eta values (fixed E)
    eta_sweep = np.array([0.25, 1.0, 4.0]) * eta
    sweep_curves = [(sigma0 / E) * (1 - np.exp(-t / (e / E))) for e in eta_sweep]

    print("  Kelvin-Voigt creep response")
    print("  " + "-"*44)
    print(f"  Young's modulus     E      = {E_kPa:>7.2f} kPa")
    print(f"  Viscosity           eta    = {eta_kPa_s:>7.2f} kPa.s")
    print(f"  Applied stress      sigma0 = {sigma0_kPa:>7.2f} kPa")
    print(f"  Retardation time    tau    = {tau:>7.3f} s")
    print(f"  Equilibrium strain  eq     = {eps_eq:>7.4f}  ({eps_eq*100:.2f} %)")
    print(f"  Strain at t = 1 s          = {(sigma0/E)*(1-np.exp(-1/tau)):>7.4f}")
    print(f"  Strain at t = tau          = {(sigma0/E)*(1-np.exp(-1)):>7.4f}   (= 63.2 % of eq.)")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    axes[0].plot(t, eps, 'b-', lw=2, label=f'tau = {tau:.2f} s')
    axes[0].axhline(eps_eq, color='green', ls='--', lw=1.2,
                    label=f'eq. strain = {eps_eq:.3f}')
    axes[0].axvline(tau, color='red', ls=':', lw=1.2, label='t = tau')
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel(r'Strain $\varepsilon(t)$')
    axes[0].set_title('Kelvin-Voigt creep')
    axes[0].legend(fontsize=8)

    colors = ['#5DADE2', '#2E86C1', '#1B4F72']
    for e, curve, c in zip(eta_sweep, sweep_curves, colors):
        axes[1].plot(t, curve, color=c, lw=1.8,
                     label=f'eta = {e/KPA:.1f} kPa.s  (tau = {e/E:.1f} s)')
    axes[1].axhline(eps_eq, color='green', ls='--', lw=1, alpha=0.6)
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel(r'Strain $\varepsilon(t)$')
    axes[1].set_title('Effect of viscosity (E fixed)')
    axes[1].legend(fontsize=8)

    axes[2].semilogx(np.clip(t, 1e-3, None), eps / eps_eq, 'b-', lw=2)
    axes[2].axhline(1 - 1/np.e, color='red', ls=':', lw=1.2, label='63.2 %')
    axes[2].axvline(tau, color='red', ls=':', lw=1.2, label='t = tau')
    axes[2].set_xlabel('Time (s, log)')
    axes[2].set_ylabel(r'$\varepsilon / \varepsilon_{eq}$')
    axes[2].set_title('Normalized creep on log time')
    axes[2].set_ylim(0, 1.05)
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

interact(
    interactive_kv_creep,
    E_kPa=FloatSlider(value=5.0, min=0.5, max=50, step=0.5, description='E (kPa)'),
    eta_kPa_s=FloatSlider(value=20, min=1, max=200, step=1, description='eta (kPa.s)'),
    sigma0_kPa=FloatSlider(value=1.0, min=0.1, max=10, step=0.1, description='sigma0 (kPa)'),
    t_max_s=FloatSlider(value=30, min=5, max=200, step=5, description='t_max (s)'),
);


---
## 2. Maxwell Stress Relaxation — Decay of Stress Under Constant Strain

The **Maxwell model** (Section 7.2.3) is a spring and a dashpot in **series**. The constitutive equation reads

$$\frac{d\varepsilon}{dt} = \frac{1}{E}\,\frac{d\sigma}{dt} + \frac{\sigma}{\eta}.$$

If we **suddenly impose** a constant strain $\varepsilon_{0}$ at $t = 0$ (an AFM **indentation-clamp** / stress-relaxation experiment), the stress decays exponentially:

$$\sigma(t) = \sigma_{0}\, e^{-t/\tau}, \qquad \sigma_{0} = E\,\varepsilon_{0}, \qquad \tau = \frac{\eta}{E}.$$

Unlike Kelvin–Voigt, the Maxwell material **flows** indefinitely — eventually $\sigma \to 0$ even though the strain is held constant.

### Exercise

The simulation imposes a step strain at $t = 0$ and reports the decaying stress, the residual stress at user-selectable times, and the characteristic time $\tau$. Numerical Problem 7.10.2 #2 ($E = 10$ kPa, $\eta = 50$ kPa·s, $\sigma_{0} = 2$ kPa) is the default.

### Tasks
1. With the defaults, verify $\tau = 5$ s and the stress at $t = \tau$ is $\sigma_{0}/e \approx 0.736$ kPa.
2. Increase $\eta$ to 200 kPa·s. Does $\sigma_{0}$ (the **initial** stress) change? Does $\tau$ change?
3. Set $E = 100$ kPa with $\eta = 50$ kPa·s. What is the new $\tau$? Which dominates the early-time response: the spring or the dashpot?
4. Compare the **half-decay time** $t_{1/2} = \tau\,\ln 2$ from the simulation with the analytical formula.

**Reflection:** *In an AFM indentation experiment on a living cell, you measure the cantilever load to decrease by 50 % over the first second after the tip stops moving. Which model parameter does this most directly constrain — $E$, $\eta$, or $\tau$?*


In [ ]:
def interactive_maxwell(E_kPa=10.0, eta_kPa_s=50.0, eps0=0.20,
                       t_max_s=20.0, n_points=400):
    """Maxwell stress-relaxation under a step strain eps0 applied at t=0."""
    E      = E_kPa * KPA
    eta    = eta_kPa_s * KPA          # Pa.s
    tau    = eta / E
    sigma0 = E * eps0                  # initial (purely elastic) response

    t        = np.linspace(0, t_max_s, n_points)
    sigma    = sigma0 * np.exp(-t / tau)
    sigma_kPa = sigma / KPA

    # sweep over eta (fixed E)
    eta_sweep   = np.array([0.25, 1.0, 4.0]) * eta
    sweep_curves = [sigma0 * np.exp(-t / (e / E)) / KPA for e in eta_sweep]

    print("  Maxwell stress relaxation")
    print("  " + "-"*44)
    print(f"  Young's modulus     E      = {E_kPa:>7.2f} kPa")
    print(f"  Viscosity           eta    = {eta_kPa_s:>7.2f} kPa.s")
    print(f"  Applied strain      eps0   = {eps0:>7.3f}")
    print(f"  Initial stress      sigma0 = {sigma0/KPA:>7.3f} kPa")
    print(f"  Relaxation time     tau    = {tau:>7.3f} s")
    print(f"  Stress at t = tau          = {sigma0*np.exp(-1)/KPA:>7.3f} kPa  (= sigma0/e)")
    print(f"  Half-decay time     t_1/2  = {tau*np.log(2):>7.3f} s")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    axes[0].plot(t, sigma_kPa, 'r-', lw=2, label=f'tau = {tau:.2f} s')
    axes[0].axhline(sigma0/KPA / np.e, color='green', ls=':', lw=1.2,
                    label='sigma0 / e')
    axes[0].axvline(tau, color='blue', ls=':', lw=1.2, label='t = tau')
    axes[0].axvline(tau*np.log(2), color='orange', ls=':', lw=1.2, label='t_1/2')
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel(r'Stress $\sigma(t)$  (kPa)')
    axes[0].set_title('Maxwell stress relaxation')
    axes[0].legend(fontsize=8)

    colors = ['#F5B041', '#E67E22', '#A04000']
    for e, curve, c in zip(eta_sweep, sweep_curves, colors):
        axes[1].plot(t, curve, color=c, lw=1.8,
                     label=f'eta = {e/KPA:.1f} kPa.s  (tau = {e/E:.1f} s)')
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel(r'Stress $\sigma(t)$  (kPa)')
    axes[1].set_title('Effect of viscosity (E fixed)')
    axes[1].legend(fontsize=8)

    axes[2].semilogy(t, sigma_kPa, 'r-', lw=2)
    axes[2].set_xlabel('Time (s)')
    axes[2].set_ylabel('Stress (kPa, log)')
    axes[2].set_title('Exponential decay on log axis')
    axes[2].grid(True, which='both', alpha=0.3)

    plt.tight_layout()
    plt.show()

interact(
    interactive_maxwell,
    E_kPa=FloatSlider(value=10, min=1, max=200, step=1, description='E (kPa)'),
    eta_kPa_s=FloatSlider(value=50, min=1, max=500, step=1, description='eta (kPa.s)'),
    eps0=FloatSlider(value=0.20, min=0.01, max=0.5, step=0.01, description='eps0'),
    t_max_s=FloatSlider(value=20, min=2, max=100, step=2, description='t_max (s)'),
);


---
## 3. Oscillatory Nanorheology — Storage Modulus $E'$, Loss Modulus $E''$ and Phase Lag $\varphi$

Section 7.4 introduces small-amplitude **oscillatory** AFM nanorheology. Apply a sinusoidal indentation

$$\delta(t) = \delta_{0} + \delta_{\text{osc}}\,\sin(\omega t),$$

and measure the force response

$$F(t) = F_{0}\,\sin(\omega t + \varphi).$$

For a viscoelastic material the response is **delayed** by a phase angle $\varphi$.  Splitting $F$ into in-phase ($\cos\varphi$) and out-of-phase ($\sin\varphi$) components defines the **storage** and **loss** moduli:

$$E' = \frac{F_{0}}{A_{c}\,\delta_{\text{osc}}}\,\cos\varphi, \qquad E'' = \frac{F_{0}}{A_{c}\,\delta_{\text{osc}}}\,\sin\varphi, \qquad E^{*} = E' + i\,E''.$$

The **loss tangent** $\tan\varphi = E''/E'$ summarizes dissipation:

| Material | $\varphi$ | $\tan\varphi$ | Behaviour |
|----------|-----------|---------------|-----------|
| Pure elastic | $0$ | $0$ | All energy stored |
| Pure viscous | $\pi/2$ | $\infty$ | All energy dissipated |
| Viscoelastic | $0 < \varphi < \pi/2$ | finite | Mixed |

The chapter's standard analog: for an SLS/Maxwell-like model with a single $\tau = \eta/E$,

$$E'(\omega) = E\,\frac{(\omega\tau)^{2}}{1+(\omega\tau)^{2}}, \qquad E''(\omega) = E\,\frac{\omega\tau}{1+(\omega\tau)^{2}}.$$

The **crossover frequency** $\omega_{c} = 1/\tau$ marks the transition from viscous-dominated ($E'' > E'$) to elastic-dominated ($E' > E''$) behaviour.

### Exercise

Pick $E$, $\eta$, and the oscillation frequency. The left panel shows the time traces (stress vs strain), the middle one extracts $E'$, $E''$ via a sine fit, and the right one sweeps the frequency to produce the full $E'(\omega)/E''(\omega)$ spectrum with the crossover marked.

### Tasks
1. Set $E = 10$ kPa, $\eta = 5$ kPa·s. What is the crossover frequency in Hz? Verify the sliders.
2. Fix the material and sweep the **drive frequency**. At what frequency is the loss tangent maximum? What is its value?
3. Numerical Problem 7.10.2 #3: $E' = 8$ kPa and $E'' = 4$ kPa. Read off the corresponding $\tan\varphi$ and $\varphi$ — the sample is more elastic or more viscous?

**Reflection:** *Why does the **same** material look "soft" at low frequencies and "stiff" at high frequencies? What experimental parameter actually controls which regime an AFM measurement samples?*


In [ ]:
def interactive_oscillatory(E_kPa=10.0, eta_kPa_s=5.0, freq_Hz=1.0,
                            delta_osc_nm=10.0, n_cycles=4):
    """Oscillatory nanorheology with an SLS/Maxwell-like single-tau spectrum."""
    E      = E_kPa * KPA
    eta    = eta_kPa_s * KPA
    tau    = eta / E
    omega  = 2 * PI * freq_Hz
    omega_c = 1 / tau                  # crossover (rad/s)
    f_c    = omega_c / (2 * PI)        # crossover (Hz)

    # Frequency-domain SLS/Maxwell moduli (per chapter eqs.)
    E_storage = E * (omega*tau)**2 / (1 + (omega*tau)**2)
    E_loss    = E * (omega*tau)      / (1 + (omega*tau)**2)
    E_complex = np.sqrt(E_storage**2 + E_loss**2)
    tan_phi   = E_loss / max(E_storage, 1e-12)
    phi       = np.arctan2(E_loss, E_storage)

    # Time traces (a few periods)
    period = 1 / freq_Hz
    t      = np.linspace(0, n_cycles * period, 1000)
    strain = delta_osc_nm * np.sin(omega * t)             # nm
    # stress response: amplitude scaled by |E*| times the strain in normalised units
    stress = (E_complex / E) * delta_osc_nm * np.sin(omega * t + phi)

    # Frequency sweep (Hz)
    freqs   = np.logspace(-3, 3, 400)
    omegas  = 2 * PI * freqs
    Ep_sw   = E * (omegas*tau)**2 / (1 + (omegas*tau)**2)
    Epp_sw  = E * (omegas*tau)    / (1 + (omegas*tau)**2)

    print("  Oscillatory nanorheology (SLS / Maxwell-like)")
    print("  " + "-"*48)
    print(f"  E = {E_kPa:.2f} kPa  |  eta = {eta_kPa_s:.2f} kPa.s  |  tau = {tau:.4f} s")
    print(f"  Drive frequency  f = {freq_Hz:.3f} Hz  (omega = {omega:.3f} rad/s)")
    print(f"  Crossover        f_c = {f_c:.3f} Hz  (E' = E'' at omega.tau = 1)")
    print()
    print(f"  Storage modulus  E'  = {E_storage/KPA:>7.2f} kPa")
    print(f"  Loss modulus     E'' = {E_loss/KPA:>7.2f} kPa")
    print(f"  Complex modulus |E*| = {E_complex/KPA:>7.2f} kPa")
    print(f"  Phase angle      phi = {np.degrees(phi):>7.2f} deg")
    print(f"  Loss tangent     tan(phi) = {tan_phi:.3f}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    ax2 = axes[0].twinx()
    axes[0].plot(t, strain, 'b-', lw=2, label='Strain (nm)')
    ax2.plot(t, stress, 'r-', lw=2, label='Stress amplitude')
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Strain (nm)', color='b')
    ax2.set_ylabel('Scaled stress', color='r')
    axes[0].set_title(f'Time response  (phi = {np.degrees(phi):.1f} deg)')
    axes[0].grid(True, alpha=0.3)

    bars = axes[1].bar(['E\'', 'E\'\'', '|E*|'],
                       [E_storage/KPA, E_loss/KPA, E_complex/KPA],
                       color=['steelblue', 'firebrick', 'gray'],
                       edgecolor='white', width=0.55)
    for b, v in zip(bars, [E_storage, E_loss, E_complex]):
        axes[1].text(b.get_x() + b.get_width()/2, v/KPA*1.02,
                     f'{v/KPA:.2f}', ha='center', fontsize=9)
    axes[1].set_ylabel('Modulus (kPa)')
    axes[1].set_title('Storage / loss / complex modulus')

    axes[2].loglog(freqs, Ep_sw/KPA,  'b-', lw=2, label="E' (storage)")
    axes[2].loglog(freqs, Epp_sw/KPA, 'r-', lw=2, label="E'' (loss)")
    axes[2].axvline(f_c, color='green', ls='--', lw=1.2,
                    label=f'f_c = {f_c:.2g} Hz')
    axes[2].axvline(freq_Hz, color='purple', ls=':', lw=1.2,
                    label=f'f = {freq_Hz:.2g} Hz')
    axes[2].set_xlabel('Frequency (Hz)')
    axes[2].set_ylabel('Modulus (kPa)')
    axes[2].set_title('Frequency sweep')
    axes[2].legend(fontsize=8)
    axes[2].grid(True, which='both', alpha=0.3)

    plt.tight_layout()
    plt.show()

interact(
    interactive_oscillatory,
    E_kPa=FloatSlider(value=10, min=0.1, max=200, step=0.1, description='E (kPa)'),
    eta_kPa_s=FloatSlider(value=5, min=0.01, max=200, step=0.1, description='eta (kPa.s)'),
    freq_Hz=FloatLogSlider(value=1.0, base=10, min=-2, max=3, step=0.1, description='f (Hz)'),
    delta_osc_nm=FloatSlider(value=10, min=1, max=100, step=1, description='delta_osc (nm)'),
    n_cycles=IntSlider(value=4, min=1, max=10, step=1, description='# cycles'),
);


---
## 4. Lissajous Curves and Energy Dissipation — Reading the Hysteresis Loop

Plotting **force vs deformation** during one full oscillation produces a **Lissajous figure** whose shape encodes the material behaviour (Section 7.3.3 & 7.4):

| Behaviour | $\varphi$ | Lissajous shape |
|-----------|-----------|------------------|
| Purely elastic | $0$ | Straight line |
| Purely viscous | $\pi/2$ | Circle (or ellipse aligned with axes) |
| Viscoelastic | $0 < \varphi < \pi/2$ | Tilted ellipse |

The **area enclosed by the loop equals the energy dissipated per cycle** (per unit volume for stress–strain):

$$W_{\text{diss}} = \oint F\,d\delta = \pi\,F_{0}\,\delta_{\text{osc}}\,\sin\varphi.$$

### Exercise

A force–deformation Lissajous loop is generated for a chosen phase angle $\varphi$. The simulation computes the loop area numerically (shoelace formula) and compares it to the analytical expression.

### Tasks
1. Set $\varphi = 0$. Why does the loop collapse to a line? What is the dissipated energy?
2. Set $\varphi = 90^\circ$. What does the loop look like? Does its area match the analytical formula?
3. Set $\varphi = 45^\circ$ and compare numerical vs analytical loop areas. Increase the number of cycles — the dissipated energy is *per cycle*; does it stay constant?
4. Numerical Problem 7.10.2 #5 (loop area of $0.5 \times 10^{-15}$ J) — choose $F_{0}$ and $\delta_{\text{osc}}$ that reproduce this, then read off the implied $\varphi$.

**Reflection:** *In AFM tapping-mode imaging, the phase image is essentially a map of the loss tangent. Why does the hysteresis loop **area** matter biologically — what does it tell you about a sample's molecular machinery?*


In [ ]:
def interactive_lissajous(F0_nN=2.0, delta_osc_nm=10.0, phi_deg=45.0,
                         n_cycles=2):
    """Lissajous figure F(delta) and hysteresis-loop area."""
    phi = np.radians(phi_deg)

    omega = 2 * PI                       # arbitrary angular frequency
    t = np.linspace(0, n_cycles * 2 * PI / omega, 2000)

    delta = delta_osc_nm * np.sin(omega * t)              # nm
    F     = F0_nN       * np.sin(omega * t + phi)         # nN

    # Closed-loop area for one cycle (shoelace on first cycle)
    one_cycle = t <= 2 * PI / omega
    x = delta[one_cycle]
    y = F[one_cycle]
    area_num = 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))
    # convert nN.nm -> J  (1 nN.nm = 1e-18 J)
    W_num_J  = area_num * 1e-18
    W_ana_J  = PI * F0_nN * delta_osc_nm * np.sin(phi) * 1e-18

    if phi_deg < 5:
        regime = "Nearly purely elastic (line)"
    elif phi_deg > 85:
        regime = "Nearly purely viscous (circle)"
    else:
        regime = "Viscoelastic (tilted ellipse)"

    print("  Lissajous loop and energy dissipation")
    print("  " + "-"*44)
    print(f"  F0          = {F0_nN:.2f} nN")
    print(f"  delta_osc   = {delta_osc_nm:.2f} nm")
    print(f"  Phase angle = {phi_deg:.1f} deg   ->  regime: {regime}")
    print(f"  tan(phi)    = {np.tan(phi):.3f}")
    print()
    print(f"  Loop area  numerical  = {W_num_J:.3e} J  per cycle")
    print(f"  Loop area  analytical = {W_ana_J:.3e} J  (= pi.F0.delta.sin phi)")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    axes[0].plot(t, delta, 'b-', lw=1.6, label='Deformation (nm)')
    axes[0].plot(t, F,     'r-', lw=1.6, label='Force (nN)')
    axes[0].set_xlabel('Time (s, normalised)')
    axes[0].set_ylabel('Signal')
    axes[0].set_title('Time traces')
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(delta, F, 'g-', lw=1.8)
    axes[1].fill(delta[one_cycle], F[one_cycle],
                 color='green', alpha=0.18, label=f'Loop area')
    axes[1].axhline(0, color='gray', ls=':', lw=0.8)
    axes[1].axvline(0, color='gray', ls=':', lw=0.8)
    axes[1].set_xlabel(r'Deformation $\delta$ (nm)')
    axes[1].set_ylabel('Force F (nN)')
    axes[1].set_title(f'Lissajous loop  (phi = {phi_deg:.0f} deg)')
    axes[1].legend(fontsize=8)
    axes[1].set_aspect('auto')

    phis = np.linspace(0, 90, 200)
    W = PI * F0_nN * delta_osc_nm * np.sin(np.radians(phis)) * 1e-18
    axes[2].plot(phis, W, 'k-', lw=2)
    axes[2].axvline(phi_deg, color='red', ls=':', lw=1.2,
                    label=f'phi = {phi_deg:.0f} deg')
    axes[2].scatter([phi_deg], [W_ana_J], color='red', s=60, zorder=5)
    axes[2].set_xlabel('Phase angle phi (deg)')
    axes[2].set_ylabel('Dissipated energy / cycle (J)')
    axes[2].set_title('Dissipation vs phase angle')
    axes[2].legend(fontsize=9)

    plt.tight_layout()
    plt.show()

interact(
    interactive_lissajous,
    F0_nN=FloatSlider(value=2.0, min=0.1, max=20, step=0.1, description='F0 (nN)'),
    delta_osc_nm=FloatSlider(value=10, min=1, max=100, step=1, description='delta (nm)'),
    phi_deg=FloatSlider(value=45, min=0, max=90, step=1, description='phi (deg)'),
    n_cycles=IntSlider(value=2, min=1, max=5, step=1, description='# cycles'),
);


---
## 5. Poroelastic Relaxation in Hydrogels — Fluid Flow, Not Solid Viscosity

Section 7.5 contrasts **viscoelasticity** (molecular rearrangement, $\tau = \eta/E$) with **poroelasticity** (fluid redistribution through a porous solid matrix, e.g. hydrogels, cartilage, cytoplasm).

The poroelastic relaxation timescale scales **diffusively** with a characteristic length $L$ and a **poroelastic diffusivity** $D_{p} = k\,E\,/\,\mu_{f}$ (permeability × modulus / fluid viscosity):

$$\boxed{\;\tau_{p} \sim \frac{L^{2}}{D_{p}}\;}$$

Crucially, **poroelastic $\tau_{p}$ depends on the probe size $L \sim \sqrt{R\delta}$**, whereas viscoelastic $\tau$ does not. This gives a clean experimental fingerprint.

### Exercise

The simulation compares a model **stress relaxation** with (i) a single-exponential viscoelastic response of timescale $\tau_{\text{ve}}$ and (ii) a poroelastic response with timescale $\tau_{p} = L^{2}/D_{p}$. The right panel sweeps the probe size $R$ (contact length $L \propto \sqrt{R\delta}$) to show how $\tau_{p}$ shifts but $\tau_{\text{ve}}$ does not.

### Tasks
1. Numerical Problem 7.10.2 #9: $L = 20\,\mu\text{m}$, $D_{p} = 5\times 10^{-11}\,\text{m}^{2}/\text{s}$. Verify $\tau_{p} = 8$ s.
2. Move from a sharp tip ($R = 25$ nm) to a colloidal probe ($R = 5\,\mu\text{m}$) at fixed $\delta$. By what factor does $\tau_{p}$ change? Why does this *not* happen for $\tau_{\text{ve}}$?
3. Cartilage has $D_{p} \approx 10^{-15}$ m²/s. With a 5-μm probe ($L \sim 1$ μm), how long must you wait for poroelastic equilibration?

**Reflection:** *If you suspect a soft sample is poroelastic, what is the single most diagnostic experiment to confirm it? (Hint: change $R$, not $\eta$.)*


In [ ]:
def interactive_poroelastic(E_kPa=20.0, eta_kPa_s=2.0,
                            R_nm=500.0, delta_nm=200.0,
                            Dp_log10_m2s=-10.0, t_max_s=20.0):
    """Compare viscoelastic vs poroelastic stress relaxation."""
    E    = E_kPa * KPA
    eta  = eta_kPa_s * KPA
    R    = R_nm * NM
    delta = delta_nm * NM
    Dp   = 10**Dp_log10_m2s              # m^2/s

    # Viscoelastic time
    tau_ve = eta / E
    # Poroelastic length scale  L ~ sqrt(R.delta)  (Hertzian contact radius)
    L      = np.sqrt(R * delta)          # m
    tau_p  = L**2 / Dp

    t = np.linspace(0, t_max_s, 600)
    sigma_ve = np.exp(-t / tau_ve)
    sigma_p  = np.exp(-t / tau_p)

    # sweep R to show probe-size dependence of tau_p
    R_sweep_nm = np.array([25, 100, 500, 2000, 10000])  # 25 nm .. 10 um
    L_sweep    = np.sqrt(R_sweep_nm * NM * delta)
    taus_p_sw  = L_sweep**2 / Dp

    print("  Viscoelastic vs poroelastic relaxation")
    print("  " + "-"*48)
    print(f"  E    = {E_kPa:.1f} kPa     eta = {eta_kPa_s:.2f} kPa.s  ->  tau_ve = {tau_ve:.3f} s")
    print(f"  R    = {R_nm:.1f} nm       delta = {delta_nm:.1f} nm")
    print(f"  L    = sqrt(R.delta) = {L*1e6:.3f} um")
    print(f"  D_p  = {Dp:.2e} m^2/s")
    print(f"  tau_p = L^2 / D_p   = {tau_p:.3f} s")
    print()
    print("  Probe-size dependence of tau_p (tau_ve is unchanged):")
    for Rval, tval in zip(R_sweep_nm, taus_p_sw):
        print(f"    R = {Rval:>6.0f} nm  ->  tau_p = {tval:>8.3f} s")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    axes[0].plot(t, sigma_ve, 'b-', lw=2, label=f'Viscoelastic  (tau_ve = {tau_ve:.2f} s)')
    axes[0].plot(t, sigma_p,  'r-', lw=2, label=f'Poroelastic    (tau_p = {tau_p:.2f} s)')
    axes[0].axhline(1/np.e, color='gray', ls=':', lw=1, label='1/e')
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel(r'Normalised stress $\sigma/\sigma_{0}$')
    axes[0].set_title('Two relaxation mechanisms')
    axes[0].legend(fontsize=8)

    axes[1].semilogx(np.clip(t, 1e-3, None), sigma_ve, 'b-', lw=2,
                     label='Viscoelastic')
    axes[1].semilogx(np.clip(t, 1e-3, None), sigma_p, 'r-', lw=2,
                     label='Poroelastic')
    axes[1].axvline(tau_ve, color='blue', ls=':', lw=1)
    axes[1].axvline(tau_p,  color='red',  ls=':', lw=1)
    axes[1].set_xlabel('Time (s, log)')
    axes[1].set_ylabel(r'$\sigma/\sigma_{0}$')
    axes[1].set_title('Relaxations on log time')
    axes[1].legend(fontsize=8)

    axes[2].loglog(R_sweep_nm, taus_p_sw, 'r-o', lw=2,
                   label='tau_p (poroelastic)')
    axes[2].axhline(tau_ve, color='blue', ls='--', lw=1.5,
                    label=f'tau_ve = {tau_ve:.2g} s (size-independent)')
    axes[2].axvline(R_nm, color='gray', ls=':', lw=1, label=f'current R = {R_nm:.0f} nm')
    axes[2].set_xlabel('Tip radius R (nm)')
    axes[2].set_ylabel('Relaxation time (s)')
    axes[2].set_title('Probe-size diagnostic')
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

interact(
    interactive_poroelastic,
    E_kPa=FloatSlider(value=20, min=1, max=200, step=1, description='E (kPa)'),
    eta_kPa_s=FloatSlider(value=2, min=0.1, max=50, step=0.1, description='eta (kPa.s)'),
    R_nm=FloatLogSlider(value=500, base=10, min=1, max=4, step=0.1, description='R (nm)'),
    delta_nm=FloatSlider(value=200, min=10, max=2000, step=10, description='delta (nm)'),
    Dp_log10_m2s=FloatSlider(value=-10, min=-15, max=-7, step=0.1,
                             description='log10(D_p)'),
    t_max_s=FloatSlider(value=20, min=2, max=200, step=2, description='t_max (s)'),
);


---
## 6. AFM Viscoelastic Force Curves — Hysteresis, Loading-Rate Dependence, and Model Fits

A real AFM **approach-retract cycle** on a viscoelastic sample exhibits two signatures (Sections 7.6 & 7.7):

* **Hysteresis** between approach and retract — energy dissipated in the cycle.
* **Loading-rate-dependent apparent stiffness** — faster indentation → stiffer-looking response.

A useful minimal model is **viscoelastic Hertz**, in which the contact force has an elastic Hertz term plus a viscous term proportional to indentation **rate**:

$$F(\delta,\dot\delta) \;=\; \frac{4}{3}\,E\,\sqrt{R}\,\delta^{3/2} \;+\; \eta_{\text{eff}}\,\sqrt{R\delta}\;\dot\delta.$$

The first term is the equilibrium Hertz; the second adds rate dependence and produces the loading–unloading hysteresis observed for cells and gels.

### Exercise

Pick the apparent modulus $E$, the effective viscosity $\eta_{\text{eff}}$, the loading rate, and the tip radius. The simulation runs an approach–retract cycle, plots both branches, and fits **plain Hertz** to the approach branch — letting you see the rate-dependent **apparent stiffening** directly.

### Tasks
1. Set $\eta_{\text{eff}} = 0$. The two branches overlap exactly (elastic limit). Verify the Hertz fit recovers $E$.
2. Numerical Problem 7.10.2 #6: stiffness at slow loading = 2.5 kPa, at fast loading = 7.5 kPa — find $E$ and $\eta_{\text{eff}}$ that reproduce this 3× ratio.
3. Increase the loading rate. Watch the loop open up. Does the equilibrium ($\dot\delta = 0$) elastic modulus change?
4. Increase $R$ from 25 nm to 1 μm at fixed $E$, $\eta_{\text{eff}}$. Does the hysteresis loop grow or shrink?

**Reflection:** *Why is "loading-rate independent" the gold standard for a clean elastic measurement? What would you do experimentally to confirm rate independence before reporting an $E$ for a cell?*


In [ ]:
def interactive_viscoelastic_curve(E_kPa=10.0, eta_eff_kPa_s=2.0, R_nm=25.0,
                                  delta_max_nm=200.0, rate_nm_s=500.0,
                                  noise_pct=2.0):
    """AFM approach-retract loop on a viscoelastic-Hertz sample, plus Hertz fit."""
    E      = E_kPa * KPA
    eta_e  = eta_eff_kPa_s * KPA            # Pa.s (effective)
    R      = R_nm * NM
    rate   = rate_nm_s * NM                 # m/s

    # symmetric triangle wave in time
    delta_m_max = delta_max_nm * NM
    t_half = delta_m_max / rate
    n = 400
    t_app = np.linspace(0, t_half, n)
    t_ret = np.linspace(t_half, 2*t_half, n)
    t     = np.concatenate([t_app, t_ret])
    delta = np.concatenate([rate * t_app,
                            rate * (2*t_half - t_ret)])
    ddelta_dt = np.concatenate([+rate * np.ones_like(t_app),
                                -rate * np.ones_like(t_ret)])

    safe_delta = np.clip(delta, 1e-12, None)
    F_elastic = (4/3) * E * np.sqrt(R) * safe_delta**1.5
    F_visc    = eta_e * np.sqrt(R * safe_delta) * ddelta_dt
    F_total   = F_elastic + F_visc

    rng = np.random.default_rng(13)
    F_obs = F_total + noise_pct/100 * np.max(F_total) * rng.normal(size=F_total.size)

    delta_nm = delta / NM
    F_nN     = F_obs / NN
    Fel_nN   = F_elastic / NN

    # split approach vs retract
    app = np.arange(0, n)
    ret = np.arange(n, 2*n)

    # Hertz fit to the approach branch (clean elastic)
    def hertz(d_nm, E_Pa):
        return (4/3) * E_Pa * np.sqrt(R) * (d_nm*NM)**1.5 / NN
    try:
        E_fit_app, = curve_fit(hertz, delta_nm[app][1:],
                               F_nN[app][1:], p0=[E])[0]
    except Exception:
        E_fit_app = np.nan
    # Hertz fit to retract for comparison
    try:
        E_fit_ret, = curve_fit(hertz, delta_nm[ret][:-1],
                               F_nN[ret][:-1], p0=[E])[0]
    except Exception:
        E_fit_ret = np.nan

    # hysteresis loop area (numerical, on observed data)
    area_nN_nm = 0.5 * abs(np.dot(delta_nm,
                                  np.roll(F_nN, -1))
                          - np.dot(F_nN, np.roll(delta_nm, -1)))
    W_J = area_nN_nm * 1e-18

    print("  Viscoelastic force curve (Hertz + viscous term)")
    print("  " + "-"*52)
    print(f"  E (true)            = {E_kPa:.2f} kPa")
    print(f"  eta_eff             = {eta_eff_kPa_s:.2f} kPa.s")
    print(f"  R = {R_nm:.0f} nm    delta_max = {delta_max_nm:.0f} nm")
    print(f"  Loading rate        = {rate_nm_s:.0f} nm/s")
    print()
    print(f"  Hertz fit approach  E_app = {E_fit_app/KPA:.2f} kPa")
    print(f"  Hertz fit retract   E_ret = {E_fit_ret/KPA:.2f} kPa")
    print(f"  Apparent stiffening (app/true) = {E_fit_app/E:.2f}x")
    print(f"  Hysteresis loop area = {W_J:.3e} J  (per cycle)")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    axes[0].plot(delta_nm[app], F_nN[app], 'b-', lw=2, label='Approach')
    axes[0].plot(delta_nm[ret], F_nN[ret], 'r-', lw=2, label='Retract')
    axes[0].plot(delta_nm, Fel_nN, 'k--', lw=1, alpha=0.6, label='Pure Hertz')
    axes[0].set_xlabel('Deformation delta (nm)')
    axes[0].set_ylabel('Force (nN)')
    axes[0].set_title(f'Approach-retract loop  (rate = {rate_nm_s:.0f} nm/s)')
    axes[0].legend(fontsize=8)

    rates_sw = np.array([100, 300, 1000, 3000, 10000])           # nm/s
    Eapp_sw  = []
    for r_nms in rates_sw:
        r = r_nms * NM
        d_app  = np.linspace(1e-9, delta_m_max, 200)
        F_app  = (4/3) * E * np.sqrt(R) * d_app**1.5                 + eta_e * np.sqrt(R * d_app) * r
        try:
            Ehat, = curve_fit(lambda d, Ep:
                              (4/3) * Ep * np.sqrt(R) * d**1.5,
                              d_app, F_app, p0=[E])[0]
            Eapp_sw.append(Ehat/KPA)
        except Exception:
            Eapp_sw.append(np.nan)

    axes[1].semilogx(rates_sw, Eapp_sw, 'k-o', lw=2)
    axes[1].axhline(E_kPa, color='green', ls='--', lw=1.5,
                    label=f'True E = {E_kPa:.1f} kPa')
    axes[1].axvline(rate_nm_s, color='purple', ls=':', lw=1.2,
                    label='current rate')
    axes[1].set_xlabel('Loading rate (nm/s)')
    axes[1].set_ylabel('Apparent E (kPa, Hertz fit to approach)')
    axes[1].set_title('Rate-dependent stiffness')
    axes[1].legend(fontsize=8)

    axes[2].plot(t*1e3, delta_nm, 'b-', lw=1.5, label='delta (nm)')
    ax2 = axes[2].twinx()
    ax2.plot(t*1e3, F_nN, 'r-', lw=1.5, label='F (nN)')
    axes[2].set_xlabel('Time (ms)')
    axes[2].set_ylabel('Deformation (nm)', color='b')
    ax2.set_ylabel('Force (nN)', color='r')
    axes[2].set_title('Time traces of cycle')

    plt.tight_layout()
    plt.show()

interact(
    interactive_viscoelastic_curve,
    E_kPa=FloatSlider(value=10, min=0.5, max=100, step=0.5, description='E (kPa)'),
    eta_eff_kPa_s=FloatSlider(value=2, min=0, max=50, step=0.1, description='eta_eff (kPa.s)'),
    R_nm=FloatLogSlider(value=25, base=10, min=1, max=3.7, step=0.1, description='R (nm)'),
    delta_max_nm=FloatSlider(value=200, min=20, max=1000, step=20, description='delta_max (nm)'),
    rate_nm_s=FloatLogSlider(value=500, base=10, min=1, max=5, step=0.1, description='rate (nm/s)'),
    noise_pct=FloatSlider(value=2, min=0, max=20, step=1, description='noise (%)'),
);


---
## 7. Frequency-Dependent Mechanics of Biological Materials — Power-Law Rheology

Living cells, cytoplasm, and the cytoskeleton do **not** obey a single $\tau$; their spectra follow **power-law rheology** over many decades (Fabry et al. 2001, *PRL*; Section 7.4.3 of the chapter):

$$G^{*}(\omega) = G_{0}\,\Big(\frac{\omega}{\omega_{0}}\Big)^{\alpha}\,\big[\cos(\pi\alpha/2) + i\,\sin(\pi\alpha/2)\big].$$

* $\alpha = 0$ → purely elastic solid (Hookean spring).
* $\alpha = 1$ → purely viscous fluid (Newtonian).
* $\alpha \approx 0.1$–$0.3$ → typical living cell ("soft glassy" rheology).
* $\alpha \approx 0.5$ → semiflexible polymer network (entropic regime).

The loss tangent of a power-law material is **frequency-independent**:

$$\tan\varphi = \tan(\pi\alpha/2).$$

### Exercise

Compare four representative biological materials by plotting their storage modulus $G'(\omega)$, loss modulus $G''(\omega)$, and loss tangent.  The default presets reproduce the **cytoplasm**, **cortical actin**, **soft hydrogel**, and an **elastic reference**.

### Tasks
1. Compute $\tan\varphi$ for $\alpha = 0.2$ (typical living cell). Verify against the formula.
2. Numerical Problem 7.10.2 #4: a hydrogel where stiffness rises from 2 kPa at 1 Hz to 10 kPa at 100 Hz. What $\alpha$ does this imply?
3. Move $\alpha$ from 0.15 (resting cell) to 0.35 (activated cell). How does the loss tangent change? What does this say about the dissipation?

**Reflection:** *Why is a power-law spectrum more biologically realistic than a single Maxwell time? What molecular picture (broad distribution of $\tau$) explains it?*


In [ ]:
def interactive_freq_dependent(G0_Pa=1000.0, alpha=0.20,
                              G0_actin=5000.0, alpha_actin=0.15,
                              G0_gel=500.0, alpha_gel=0.05,
                              show_elastic=True):
    """Power-law rheology spectra for several biological-material presets."""
    omega = np.logspace(-2, 4, 400)              # rad/s
    omega0 = 1.0                                 # 1 rad/s reference

    def power_law(G0, alpha):
        Gstar = G0 * (omega/omega0)**alpha
        Gp    = Gstar * np.cos(PI*alpha/2)
        Gpp   = Gstar * np.sin(PI*alpha/2)
        return Gp, Gpp

    Gp_c,   Gpp_c   = power_law(G0_Pa,    alpha)
    Gp_a,   Gpp_a   = power_law(G0_actin, alpha_actin)
    Gp_g,   Gpp_g   = power_law(G0_gel,   alpha_gel)
    if show_elastic:
        Gp_el  = np.full_like(omega, 10000.0)
        Gpp_el = np.full_like(omega, 1.0)

    tan_phi_c = np.tan(PI*alpha/2)
    tan_phi_a = np.tan(PI*alpha_actin/2)
    tan_phi_g = np.tan(PI*alpha_gel/2)

    print("  Power-law rheology of biological materials")
    print("  " + "-"*48)
    print(f"  Cytoplasm        : G0 = {G0_Pa:>6.0f} Pa  alpha = {alpha:.2f}  "
          f"tan(phi) = {tan_phi_c:.3f}")
    print(f"  Cortical actin   : G0 = {G0_actin:>6.0f} Pa  alpha = {alpha_actin:.2f}  "
          f"tan(phi) = {tan_phi_a:.3f}")
    print(f"  Soft hydrogel    : G0 = {G0_gel:>6.0f} Pa  alpha = {alpha_gel:.2f}  "
          f"tan(phi) = {tan_phi_g:.3f}")
    print(f"  Elastic ref      : alpha = 0 (no frequency dependence)")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    axes[0].loglog(omega, Gp_c, 'b-', lw=2, label=f"Cytoplasm (alpha={alpha:.2f})")
    axes[0].loglog(omega, Gp_a, 'r-', lw=2, label=f"Actin (alpha={alpha_actin:.2f})")
    axes[0].loglog(omega, Gp_g, 'g-', lw=2, label=f"Hydrogel (alpha={alpha_gel:.2f})")
    if show_elastic:
        axes[0].loglog(omega, Gp_el, 'k--', lw=1.5, label='Elastic ref')
    axes[0].set_xlabel('omega (rad/s)')
    axes[0].set_ylabel("G' (Pa)")
    axes[0].set_title("Storage modulus")
    axes[0].legend(fontsize=8)
    axes[0].grid(True, which='both', alpha=0.3)

    axes[1].loglog(omega, Gpp_c, 'b-', lw=2, label=f"Cytoplasm")
    axes[1].loglog(omega, Gpp_a, 'r-', lw=2, label=f"Actin")
    axes[1].loglog(omega, Gpp_g, 'g-', lw=2, label=f"Hydrogel")
    if show_elastic:
        axes[1].loglog(omega, Gpp_el, 'k--', lw=1.5, label='Elastic ref')
    axes[1].set_xlabel('omega (rad/s)')
    axes[1].set_ylabel("G'' (Pa)")
    axes[1].set_title("Loss modulus")
    axes[1].legend(fontsize=8)
    axes[1].grid(True, which='both', alpha=0.3)

    axes[2].semilogx(omega, Gpp_c/Gp_c, 'b-', lw=2, label='Cytoplasm')
    axes[2].semilogx(omega, Gpp_a/Gp_a, 'r-', lw=2, label='Actin')
    axes[2].semilogx(omega, Gpp_g/Gp_g, 'g-', lw=2, label='Hydrogel')
    if show_elastic:
        axes[2].semilogx(omega, Gpp_el/Gp_el, 'k--', lw=1.5, label='Elastic ref')
    axes[2].set_xlabel('omega (rad/s)')
    axes[2].set_ylabel('Loss tangent  tan(phi) = G\"/G\'')
    axes[2].set_title('Loss tangent (power-law: flat)')
    axes[2].legend(fontsize=8)
    axes[2].grid(True, which='both', alpha=0.3)

    plt.tight_layout()
    plt.show()

interact(
    interactive_freq_dependent,
    G0_Pa=FloatSlider(value=1000, min=100, max=20000, step=100,
                      description='G0 cyto (Pa)'),
    alpha=FloatSlider(value=0.20, min=0.0, max=0.6, step=0.01,
                      description='alpha cyto'),
    G0_actin=FloatSlider(value=5000, min=500, max=50000, step=500,
                         description='G0 actin'),
    alpha_actin=FloatSlider(value=0.15, min=0.0, max=0.6, step=0.01,
                            description='alpha actin'),
    G0_gel=FloatSlider(value=500, min=50, max=10000, step=50,
                       description='G0 gel (Pa)'),
    alpha_gel=FloatSlider(value=0.05, min=0.0, max=0.6, step=0.01,
                          description='alpha gel'),
    show_elastic=Checkbox(value=True, description='Show elastic ref'),
);


---
## 8. Living Cell Dynamic Mechanics — Passive vs Active Fluctuations (ATP-Dependence)

Living cells are **non-equilibrium**: in addition to thermal (passive) viscoelastic fluctuations, the actomyosin cytoskeleton generates **active** ATP-driven force fluctuations that **violate the fluctuation–dissipation theorem (FDT)** (Section 7.6.4 & 7.8.8).

A useful experimental decomposition (Brangwynne, MacKintosh, Weitz):

* **Passive thermal**:  cantilever-position power spectrum scales as $1/\omega$ for a viscoelastic medium.
* **Active**:  adds a low-frequency $1/\omega^{2}$ component proportional to ATP-driven activity, vanishing in fixed (dead) cells.

The simulation below produces a synthetic cantilever-fluctuation power spectrum $S(\omega)$ for three conditions:

1. **Fixed cell**  — purely passive viscoelastic spectrum.
2. **Living cell** — passive + active power.
3. **ATP-depleted living cell** — intermediate.

### Exercise

Vary the activity strength and cytoplasmic stiffness, and observe (i) the broad-band excess power at low frequency in the living spectrum and (ii) how the **active/passive ratio** evolves with frequency.

### Tasks
1. Set activity strength to 0. The "living" and "fixed" curves overlap — equilibrium is restored.
2. Increase activity strength. Where in the spectrum does the active excess dominate — low or high frequencies?
3. Compare the **active/passive ratio** for $\omega = 1$ rad/s vs $\omega = 1000$ rad/s.  Why is the low-frequency window where active signatures are easiest to see?

**Reflection:** *Active force fluctuations carry information that no purely viscoelastic model can. What biological insight (e.g. metabolic state, drug response) does measuring an AFM "active spectrum" give you that a static $E^{*}$ cannot?*


In [ ]:
def interactive_active_cell(G0_Pa=1000.0, alpha=0.20,
                           activity_strength=2.0, kBT=4.1e-21,
                           atp_level=1.0):
    """Cantilever fluctuation spectrum: passive viscoelastic + active ATP component."""
    omega = np.logspace(-1, 4, 400)         # rad/s
    omega0 = 1.0

    Gstar = G0_Pa * (omega/omega0)**alpha
    Gp    = Gstar * np.cos(PI*alpha/2)
    Gpp   = Gstar * np.sin(PI*alpha/2)
    G_abs2 = Gp**2 + Gpp**2

    # Passive (FDT) PSD (arbitrary tip-coupling constant)
    coupling = 1e-6                          # arbitrary, only relative changes matter
    S_passive = (2 * kBT / omega) * (Gpp / G_abs2) / coupling

    # Active excess: stronger at low omega, scales with ATP
    S_active  = activity_strength * atp_level                 * (G0_Pa / G_abs2)                 * (1.0 / omega**2) * 1e-3

    S_living  = S_passive + S_active
    S_fixed   = S_passive
    S_atpdep  = S_passive + 0.1 * S_active

    ratio = S_active / np.maximum(S_passive, 1e-30)

    print("  Passive vs active fluctuation spectra")
    print("  " + "-"*44)
    print(f"  G0 = {G0_Pa:.0f} Pa,  alpha = {alpha:.2f},  activity = {activity_strength:.1f}")
    print(f"  Loss tangent (passive)  = {np.tan(PI*alpha/2):.3f}")
    idx_low  = np.argmin(np.abs(omega - 1))
    idx_high = np.argmin(np.abs(omega - 1000))
    print(f"  Active/passive ratio @ omega=   1 rad/s : {ratio[idx_low]:.2f}")
    print(f"  Active/passive ratio @ omega=1000 rad/s : {ratio[idx_high]:.2g}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    axes[0].loglog(omega, S_fixed,  'k-', lw=2, label='Fixed (passive)')
    axes[0].loglog(omega, S_atpdep, 'orange', lw=2, label='ATP-depleted')
    axes[0].loglog(omega, S_living, 'r-', lw=2, label='Living')
    axes[0].set_xlabel('omega (rad/s)')
    axes[0].set_ylabel('Fluctuation PSD  S(omega) (a.u.)')
    axes[0].set_title('Cantilever fluctuation spectra')
    axes[0].legend(fontsize=8)
    axes[0].grid(True, which='both', alpha=0.3)

    axes[1].loglog(omega, S_passive, 'b-', lw=2, label='Passive (FDT)')
    axes[1].loglog(omega, S_active,  'g-', lw=2, label='Active excess')
    axes[1].set_xlabel('omega (rad/s)')
    axes[1].set_ylabel('PSD component (a.u.)')
    axes[1].set_title('Decomposition: passive vs active')
    axes[1].legend(fontsize=8)
    axes[1].grid(True, which='both', alpha=0.3)

    axes[2].loglog(omega, ratio, 'm-', lw=2)
    axes[2].axhline(1, color='gray', ls=':', lw=1)
    axes[2].set_xlabel('omega (rad/s)')
    axes[2].set_ylabel('Active / Passive')
    axes[2].set_title('Where activity dominates')
    axes[2].grid(True, which='both', alpha=0.3)

    plt.tight_layout()
    plt.show()

interact(
    interactive_active_cell,
    G0_Pa=FloatSlider(value=1000, min=100, max=20000, step=100, description='G0 (Pa)'),
    alpha=FloatSlider(value=0.20, min=0.0, max=0.5, step=0.01, description='alpha'),
    activity_strength=FloatSlider(value=2.0, min=0, max=10, step=0.1,
                                  description='activity'),
    atp_level=FloatSlider(value=1.0, min=0.0, max=2.0, step=0.05,
                          description='ATP level'),
);


---

## Summary

| Exercise | Section | Key concept | Take-home |
|---|---|---|---|
| 1 | 7.2.2, 7.3.1 | Kelvin–Voigt creep | Retardation time $\tau=\eta/E$; equilibrium strain $\sigma_{0}/E$ |
| 2 | 7.2.3, 7.3.2 | Maxwell stress relaxation | Stress decays exponentially with $\tau=\eta/E$ |
| 3 | 7.4 | Oscillatory $E'/E''$ | Crossover at $\omega\tau=1$; loss tangent $=\tan\varphi$ |
| 4 | 7.3.3, 7.4 | Lissajous loops | Loop area $=\pi F_{0}\delta_{0}\sin\varphi$ = dissipated energy/cycle |
| 5 | 7.5 | Poroelasticity | $\tau_{p}\sim L^{2}/D_{p}$ depends on probe size; viscoelastic $\tau$ does not |
| 6 | 7.6, 7.7 | Rate-dependent AFM curves | Hysteresis + apparent stiffening grow with loading rate |
| 7 | 7.4.3 | Power-law rheology | Cells obey $G^{*}\propto\omega^{\alpha}$; living cells $\alpha\sim 0.1$–$0.3$ |
| 8 | 7.6.4, 7.8.8 | Active fluctuations | ATP-driven excess violates FDT; vanishes in fixed cells |

**Key chapter take-aways:**
- Real biological materials are **viscoelastic** — both **storage** ($E'$) and **loss** ($E''$) moduli matter, and depend on **timescale**.
- AFM provides creep, stress-relaxation, hysteresis, and oscillatory tools — each probes a different facet of the same underlying complex modulus $E^{*}(\omega)$.
- **Poroelasticity** (fluid flow) and **viscoelasticity** (molecular friction) look similar in a single experiment but separate cleanly when probe size is varied.
- **Power-law rheology** describes cells across many decades of frequency; the exponent $\alpha$ is a compact mechanobiological phenotype.
- **Active matter physics** is essential to interpret living-cell mechanics — equilibrium FDT-based analysis is not enough.
- Quantitative AFM viscoelastic measurements demand **explicit reporting** of loading rate, frequency window, probe geometry, temperature, and biological condition (ATP, fixation, drug).

---

*End of Chapter 7 exercises. Chapter 8 will extend AFM mechanics to single-molecule force spectroscopy and bond-rupture statistics.*
